In [2]:
import pandas as pd

dftransactions = pd.read_csv("LI-Small_Trans.csv")

dfaccounts = pd.read_csv("LI-Small_accounts.csv")

print(dftransactions.head())
#print(dfaccounts.head())


          Timestamp  From Bank    Account  To Bank  Account.1  \
0  2022/09/01 00:08         11  8000ECA90       11  8000ECA90   
1  2022/09/01 00:21       3402  80021DAD0     3402  80021DAD0   
2  2022/09/01 00:00         11  8000ECA90     1120  8006AA910   
3  2022/09/01 00:16       3814  8006AD080     3814  8006AD080   
4  2022/09/01 00:00         20  8006AD530       20  8006AD530   

   Amount Received Receiving Currency  Amount Paid Payment Currency  \
0       3195403.00          US Dollar   3195403.00        US Dollar   
1          1858.96          US Dollar      1858.96        US Dollar   
2        592571.00          US Dollar    592571.00        US Dollar   
3            12.32          US Dollar        12.32        US Dollar   
4          2941.56          US Dollar      2941.56        US Dollar   

  Payment Format  Is Laundering  
0   Reinvestment              0  
1   Reinvestment              0  
2         Cheque              0  
3   Reinvestment              0  
4   Reinvest

I - Profiling

In [3]:
dfaccounts.dtypes

Bank Name         object
Bank ID            int64
Account Number    object
Entity ID         object
Entity Name       object
dtype: object

In [4]:
dftransactions.dtypes

Timestamp              object
From Bank               int64
Account                object
To Bank                 int64
Account.1              object
Amount Received       float64
Receiving Currency     object
Amount Paid           float64
Payment Currency       object
Payment Format         object
Is Laundering           int64
dtype: object

In [5]:
dftransactions = dftransactions.drop(columns = "Is Laundering")

Now I am going to count distinct and unique values for each column of the transactions table.
Here we have to be careful because "unique" in Pandas equals "distinct" in PowerBI.

In [6]:
# Distinct values

print(dftransactions.nunique())




Timestamp               14533
From Bank               41814
Account                681281
To Bank                 21588
Account.1              576176
Amount Received       1194921
Receiving Currency         15
Amount Paid           1204309
Payment Currency           15
Payment Format              7
dtype: int64


So for obtaining unique values (i.e. values in a column that appear just once), I should get first the value counts and then count how many values are "repeated" just one time.

In [7]:
ValueCounts= {}

for i in dftransactions:
    ValueCounts[i] = dftransactions[i].value_counts()
  


In [8]:
print(ValueCounts)

{'Timestamp': Timestamp
2022/09/01 00:22    15221
2022/09/01 00:20    15070
2022/09/01 00:01    15062
2022/09/01 00:21    15061
2022/09/01 00:14    15049
                    ...  
2022/09/11 14:17        1
2022/09/15 15:44        1
2022/09/16 00:00        1
2022/09/16 12:41        1
2022/09/11 04:20        1
Name: count, Length: 14533, dtype: int64, 'From Bank': From Bank
70        609991
11        123456
20        120114
14         64681
12         64465
           ...  
367062         1
368024         1
362588         1
372033         1
372462         1
Name: count, Length: 41814, dtype: int64, 'Account': Account
10042B660    222037
10042B6A8    138777
10042B6F0     42385
10042B780     30802
10042BA51     28932
              ...  
8140FF750         1
81402A9A0         1
814018620         1
813FD9E30         1
81B3D58E1         1
Name: count, Length: 681281, dtype: int64, 'To Bank': To Bank
11        66055
20        56805
14        36313
12        34960
18        34178
          ...  

In [9]:
print((ValueCounts["Timestamp"] == 1))

Timestamp
2022/09/01 00:22    False
2022/09/01 00:20    False
2022/09/01 00:01    False
2022/09/01 00:21    False
2022/09/01 00:14    False
                    ...  
2022/09/11 14:17     True
2022/09/15 15:44     True
2022/09/16 00:00     True
2022/09/16 12:41     True
2022/09/11 04:20     True
Name: count, Length: 14533, dtype: bool


In [10]:
uniqueValues = {}
for i in ValueCounts: #every i is a key from a dictionary and its value is a dataframe that can be manipulated with Pandas
    uniqueValues[i] = ValueCounts[i][ValueCounts[i] == 1].count()


In [11]:
print(uniqueValues)

{'Timestamp': np.int64(53), 'From Bank': np.int64(4714), 'Account': np.int64(211825), 'To Bank': np.int64(5156), 'Account.1': np.int64(157062), 'Amount Received': np.int64(615389), 'Receiving Currency': np.int64(0), 'Amount Paid': np.int64(621441), 'Payment Currency': np.int64(0), 'Payment Format': np.int64(0)}


Checking if everytime a bank name is repeated, the same ID is assigned to that bank.

In [40]:
bankNamesGroupinbAndCountingValues = dfaccounts.groupby("Bank Name")["Bank ID"].nunique()
bankNamesGroupinbAndCountingValues

Bank Name
Acme Bancorp             22
Acme Bank                15
Acme Community Bank      19
Acme Cooperative Bank    15
Acme Credit Union        14
                         ..
Willows Credit Union     15
Willows Federal Bank     28
Willows Savings Bank     15
Willows Thrift           17
Willows Trust Bank        8
Name: Bank ID, Length: 27652, dtype: int64

In [53]:
bankNamesGroupinbAndCountingValues[bankNamesGroupinbAndCountingValues != 1].count()

np.int64(468)

Looking closer at Arbor Bank different IDs.

In [ ]:
x = dfaccounts[dfaccounts["Bank Name"] == "Arbor Bank"]["Bank ID"].value_counts()
x

Now I want to count values in the Bank Name column from the Accounts table to check if the relation Bank ID → Bank Name is a function.

In [ ]:
bankIDGrouping = dfaccounts.groupby("Bank ID")["Bank Name"].nunique()
bankIDGrouping[bankIDGrouping != 1].count

np.int64(0)

Checking which rows are related to more than one account number.

In [56]:
a = dfaccounts["Account Number"].value_counts()

b = a[a != 1]

b

Account Number
8177C94B0    2
817038DF0    2
8177C8ED0    2
817037B20    2
Name: count, dtype: int64

Selecting all the rows of the dataframe which have repeated account numbers.

In [57]:
c = b.index
repeatedAccountNumbers = dfaccounts[dfaccounts["Account Number"].isin(c)]

repeatedAccountNumbers

,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
364085,Savings Bank of the South,61144,817037B20,8007D0D30,Corporation #54169
434859,Savings Bank of the South,61144,817038DF0,80100DA30,Partnership #788
547803,Spain Bank #507,144720,8177C8ED0,800C73BF0,Corporation #38102
555545,Spain Bank #507,144720,8177C94B0,800981A00,Partnership #32750
569902,Savings Bank of Columbus,113213,817037B20,80129EC80,Corporation #795
571683,Estonia Bank #2218,144840,8177C8ED0,800B01A00,Sole Proprietorship #31292
573683,Estonia Bank #2218,144840,8177C94B0,800BB3280,Corporation #35681
585843,Savings Bank of Columbus,113213,817038DF0,8014110E0,Corporation #11573


Checking duplicated accounts.

In [ ]:
repeatedAccountNumbers[["Bank ID","Account Number"]].duplicated()

364085    False
434859    False
547803    False
555545    False
569902    False
571683    False
573683    False
585843    False
dtype: bool

Checking Entity ID and Entity Name: 224,931 distinct and 145,348 unique values.
These two columns share the same amount of distinct and unique values.
I.e.: is the relation between these columns a bijective function?
To check this I would have to see with how many names each ID is related.

In [59]:
dfaccounts["Entity ID"]

0         800D8CCF0
1         800B505E0
2         800D03F60
3         801567C10
4         801085E00
            ...    
712683    800D080A0
712684    8005319C0
712685    800453480
712686    8003B5120
712687    8003122E0
Name: Entity ID, Length: 712688, dtype: object

Checking:

1) if Payment Currency ≠ Receiving Currency then Amount Paid ≠ Amount Received

2) if Payment Currency = Receiving Currency then Amount Paid = Amount Received


In [60]:
sameCurrency = dftransactions[dftransactions["Payment Currency"] == dftransactions["Receiving Currency"]][["Amount Paid","Amount Received"]]

#sameCurrency

#sameCurrency has 6,825,173 / 6,924,049 rows

sameCurrency[sameCurrency["Amount Paid"] == sameCurrency["Amount Received"]] #it has 6,825,173 rows, so in all of them both attributes are equal.

,Amount Paid,Amount Received
0,3.195403e+06,3.195403e+06
1,1.858960e+03,1.858960e+03
2,5.925710e+05,5.925710e+05
3,1.232000e+01,1.232000e+01
4,2.941560e+03,2.941560e+03
...,...,...
6924044,3.346900e-02,3.346900e-02
6924045,1.313000e-03,1.313000e-03
6924046,1.305800e-02,1.305800e-02
6924047,4.145370e-01,4.145370e-01


In [61]:
difCurrency = dftransactions[dftransactions["Payment Currency"] != dftransactions["Receiving Currency"]][["Timestamp", "Payment Currency", "Receiving Currency","Amount Paid","Amount Received"]]
difCurrency
#difCurrency.shape

# difCurrency has 98,876 / 6,924,049 rows.
# 6,825,173 + 98,876 = 6,924,049 rows.

difCurrency[difCurrency["Amount Paid"] != difCurrency["Amount Received"]] #it has 98,858 rows, which means there are 18 rows with different currencies and equal Amount Paid, Amount Received.

,Timestamp,Payment Currency,Receiving Currency,Amount Paid,Amount Received
2770,2022/09/01 00:12,US Dollar,Euro,55.79,47.610000
8081,2022/09/01 00:28,US Dollar,Yuan,142.53,954.620000
10451,2022/09/01 00:18,US Dollar,Yen,160.63,16930.030000
12948,2022/09/01 00:17,US Dollar,UK Pound,18.76,14.520000
13799,2022/09/01 00:02,US Dollar,Euro,43.35,37.000000
...,...,...,...,...,...
6924007,2022/09/10 23:57,Yuan,Bitcoin,0.39,0.000005
6924009,2022/09/10 23:30,Yuan,Bitcoin,0.55,0.000007
6924019,2022/09/10 23:38,US Dollar,Bitcoin,0.08,0.000007
6924021,2022/09/10 23:31,US Dollar,Bitcoin,0.23,0.000020


Let's see those 18 rows with different currency and equal amounts

In [62]:

difCurrency[difCurrency["Amount Paid"] == difCurrency["Amount Received"]][["Timestamp", "Payment Currency", "Receiving Currency", "Amount Paid","Amount Received"]]

,Timestamp,Payment Currency,Receiving Currency,Amount Paid,Amount Received
89998,2022/09/01 00:15,UK Pound,US Dollar,0.03,0.03
137951,2022/09/01 00:08,Canadian Dollar,US Dollar,0.01,0.01
162050,2022/09/01 00:18,UK Pound,US Dollar,0.01,0.01
236263,2022/09/01 00:13,UK Pound,Euro,0.02,0.02
314776,2022/09/01 00:07,Rupee,Yen,0.01,0.01
469282,2022/09/01 00:51,Yuan,Saudi Riyal,0.01,0.01
653040,2022/09/01 04:32,Canadian Dollar,Saudi Riyal,0.01,0.01
728900,2022/09/01 06:44,Swiss Franc,US Dollar,0.01,0.01
841429,2022/09/01 09:14,Canadian Dollar,US Dollar,0.02,0.02
858188,2022/09/01 09:24,Brazil Real,Shekel,0.01,0.01


We can see that most of these transactions (#16) have been done during the same day (2022/09/01) while 2 of them have been done in different months (february and august). 
But we can also see that the amounts are really low, so the explanation to this observation is probably that all involve amounts equal or lower than 0.03, 
consistent with rounding at negligible values. Not material to the analysis.



In [63]:
print(dfaccounts.columns)
dfaccounts.head()

Index(['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name'], dtype='object')


,Bank Name,Bank ID,Account Number,Entity ID,Entity Name
0,China Bank #2820,314693,81B86A280,800D8CCF0,Corporation #41344
1,France Bank #4585,311253,8187FEA80,800B505E0,Corporation #54497
2,China Bank #2242,39996,803961E00,800D03F60,Partnership #36904
3,National Bank of Newport,331440,81B075800,801567C10,Corporation #16224
4,UK Bank #33,135417,80CF87C80,801085E00,Partnership #72930


Primary key - Accounts table

Now I am going to check if the account table's primary key is different in every row.

In [64]:
AccountsPrimaryKey = dfaccounts.duplicated(subset = ["Bank ID","Account Number"])

AccountsPrimaryKey[AccountsPrimaryKey != False]

Series([], dtype: bool)

As we can see, there are no results when we look for True values, meaning there are no repeated rows regarding these two attributes.

Primary key - Transactions table

Since there is no natural primary key in dftransactions, I will create a synthetic one.

In [66]:
dftransactions = dftransactions.reset_index(names="TransactionID")

ValueError: cannot insert TransactionID, already exists

Checking the relation dfaccounts.Entity ID --> dfaccounts.Entity Names

In [ ]:
EntitiesIDGrouping = dfaccounts.groupby("Entity ID")["Entity Name"].nunique()

EntitiesIDGrouping[EntitiesIDGrouping != 1]

Series([], Name: Entity Name, dtype: int64)

This shows that every ID is related with one Entity Name, which is what we suspected just by looking at the first distinct and unique values observations.

Cross-table referential integrity check

1) (dftransactions.From Bank, dftrasactions.Account) → (dfaccounts.Bank ID, dfaccounts.Account Number)

In [ ]:
mergedTable1 = dftransactions.merge(dfaccounts, left_on = ["From Bank","Account"], right_on = ["Bank ID","Account Number"], how = "left")

mergedTable1[["Bank ID","Account Number"]].isna().any()


Bank ID           False
Account Number    False
dtype: bool

Cross-table referential integrity check

2) (dftransactions.To Bank, dftrasactions.Account1) → (dfaccounts.Bank ID, dfaccounts.Account Number)

In [ ]:
mergedTable2 = dftransactions.merge(dfaccounts, left_on = ["To Bank","Account.1"], right_on = ["Bank ID","Account Number"], how = "left")

mergedTable2[["Bank ID","Account Number"]].isna().any()



Bank ID           False
Account Number    False
dtype: bool

II - Cleaning


Checking if there are empty rows in dftransactions

In [68]:
dftransactions.isna().any().any()

np.False_

Renaming columns with composite names separated by a space

In [89]:
dfaccounts = dfaccounts.rename(columns={
    "Bank Name": "bank_name",
    "Bank ID": "bank_id",
    "Account Number": "account_number",
    "Entity ID": "entity_id",
    "Entity Name": "entity_name"
})

dftransactions = dftransactions.rename(columns={
    "TransactionID": "id",
    "From Bank": "from_bank",
    "To Bank": "to_bank",
    "Account.1": "to_account",
    "Amount Received": "amount_received",
    "Receiving Currency": "receiving_currency",
    "Amount Paid": "amount_paid",
    "Payment Format": "payment_format",
    "Timestamp": "timestamp",
    "Account": "account",
    "Payment Currency": "payment_currency"
})


Converting data type in the dfaccounts table

In [99]:
dfaccounts["bank_id"] = dfaccounts["bank_id"].astype(str)

Converting data type in the dfaccounts table

In [100]:
dftransactions["from_bank"] = dftransactions["from_bank"].astype(str)
dftransactions["to_bank"] = dftransactions["to_bank"].astype(str)

In [101]:
dftransactions["timestamp"] = pd.to_datetime(dftransactions["timestamp"])

In [107]:
dftransactions["id"] = dftransactions["id"].astype(str)

Checking duplicates

In [123]:
print(dfaccounts.duplicated().any())
print(dftransactions.drop(columns="id").duplicated().any())

False
True


In [143]:
dftransactions[dftransactions.drop(columns="id").duplicated(keep = False)].count()

id                    16
timestamp             16
from_bank             16
account               16
to_bank               16
to_account            16
amount_received       16
receiving_currency    16
amount_paid           16
payment_currency      16
payment_format        16
dtype: int64

Cleaning Final Validation

In [144]:
dfaccounts.columns

Index(['bank_name', 'bank_id', 'account_number', 'entity_id', 'entity_name'], dtype='object')

In [145]:
dftransactions.columns

Index(['id', 'timestamp', 'from_bank', 'account', 'to_bank', 'to_account',
       'amount_received', 'receiving_currency', 'amount_paid',
       'payment_currency', 'payment_format'],
      dtype='object')

In [146]:
dfaccounts.dtypes


bank_name         object
bank_id           object
account_number    object
entity_id         object
entity_name       object
dtype: object

In [147]:
dftransactions.dtypes

id                            object
timestamp             datetime64[ns]
from_bank                     object
account                       object
to_bank                       object
to_account                    object
amount_received              float64
receiving_currency            object
amount_paid                  float64
payment_currency              object
payment_format                object
dtype: object

In [148]:
dfaccounts.shape

(712688, 5)

In [149]:
dftransactions.shape

(6924049, 11)